# Beobachter am Beispiel IMU und Encoder

Jonas Frei, 15.10.2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Encoder

Erzeugen eines Encodersignals

In [ ]:
dt = 0.001
t = np.arange(0, 4 + dt/2, dt)
l = len(t)
x = np.sin(2*np.pi*t)

Diskretisieren des Signals

In [ ]:
def discretize(x, bins):
    """Hilfsfunktion zur Diskretisierung von kontinuierlichen Daten.

    Einteilung in `bins` gleich breite Intervalle über [min(x), max(x)],
    Rückgabe des 1-basierten Bin-Index."""
    xmin, xmax = x.min(), x.max()
    idx = np.floor((x - xmin)/(xmax - xmin)*bins)
    idx = np.clip(idx, 0, bins - 1)
    return (idx + 1).astype(int)


bins = 500
x_disc = (discretize(x, bins) - 1)/(bins - 1)*2 - 1

Berechnen der Geschwindigkeit durch Ableiten

In [ ]:
xd = np.concatenate(([0], np.diff(x)/dt))
xd_disc = np.concatenate(([0], np.diff(x_disc)/dt))

Plotten

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(9, 7))

axs[0].plot(t, x_disc, linewidth=1.5, label="Diskretisiert")
axs[0].plot(t, x, linewidth=1.5, label="Echtes Signal")
axs[0].set_title("Mit Encoder gemessene Position")
axs[0].legend()
axs[0].set_xlabel("t [s]")
axs[0].set_ylabel("x [m]")

axs[1].plot(t, xd_disc, linewidth=1.5, label="Diskretisiert")
axs[1].plot(t, xd, linewidth=1.5, label="Echtes Signal")
axs[1].set_title("Daraus abgeleitete Geschwindigkeit")
axs[1].legend()
axs[1].set_xlabel("t [s]")
axs[1].set_ylabel("dx/dt [m/s]")

plt.tight_layout()
plt.show()

## IMU

Erzeugen eines IMU-Signals

In [ ]:
acc = np.concatenate(([0, 0], np.diff(np.diff(x)/dt)/dt))

Hinzufügen von Rauschen und Offset

In [ ]:
offset = -1
acc_noise = acc + np.random.randn(l) + offset

Berechnen der Geschwindigkeit durch Integrieren

In [ ]:
vel = np.zeros(l)
vel_noise = np.zeros(l)

for i in range(1, l):
    vel[i] = vel[i - 1] + acc[i - 1]*dt
    vel_noise[i] = vel_noise[i - 1] + acc_noise[i - 1]*dt

Plotten

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(9, 7))

axs[0].plot(t, acc_noise, linewidth=1.5, label="Rauschen und Offset")
axs[0].plot(t, acc, linewidth=1.5, label="Echtes Signal")
axs[0].set_title("Beschleunigung")
axs[0].legend()
axs[0].set_xlabel("t [s]")
axs[0].set_ylabel("dx/dt^2 [m/s^2]")

axs[1].plot(t, vel_noise, linewidth=1.5, label="Rauschen und Offset")
axs[1].plot(t, vel, linewidth=1.5, label="Echtes Signal")
axs[1].set_title("Geschwindigkeit")
axs[1].legend()
axs[1].set_xlabel("t [s]")
axs[1].set_ylabel("dx/dt [m/s]")

plt.tight_layout()
plt.show()

## Problematik zusammengefasst

Das Encodersignal ist aufgrund der Diskretisierung sehr sprunghaft, während das Signal der IMU einen Offset aufweist und über die Zeit davon driftet.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, xd_disc, linewidth=1.5, label="Encoder")
ax.plot(t, vel_noise, linewidth=1.5, label="IMU")
ax.plot(t, xd, linewidth=1.5, label="Echtes Signal")
ax.set_title("Problematik IMU und Encoder")
ax.set_xlabel("t [s]")
ax.set_ylabel("dx/dt [m/s]")
ax.legend(loc="best")
plt.show()

## Beobachter

Parameter definieren

In [ ]:
tau = np.array([0.005, 0.5, 10])
omega0 = 1/tau
D = 0.7
KP = omega0**2
KD = 2*D*omega0

Variablen initialisieren

In [ ]:
vel_o = 5*np.ones((len(tau), l))

Beobachteralgorithmus und Plotten

Die Beobachterzustände (`acc_o`, `pos_o`, `e_o`, `e_o_last`, `acc_co`) werden für jedes τ neu auf null gesetzt, damit die Simulation für jedes τ unabhängig und mit denselben Anfangsbedingungen startet.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for i in range(len(tau)):
    acc_o = 0.0
    pos_o = 0.0
    e_o = 0.0
    e_o_last = 0.0
    acc_co = 0.0

    for j in range(1, l):
        acc_o = acc[j - 1] + acc_co
        vel_o[i, j] = vel_o[i, j - 1] + acc_o*dt
        pos_o = pos_o + vel_o[i, j - 1]*dt
        e_o = x_disc[j - 1] - pos_o
        acc_co = KP[i]*e_o + KD[i]*(e_o - e_o_last)/dt
        e_o_last = e_o
    ax.plot(t, vel_o[i, :], linewidth=1.5, label=f"tau = {tau[i]}s")

ax.plot(t, xd, linewidth=1.5, label="Echtes Signal")
ax.set_title("Fusionierte Geschwindigkeit")
ax.set_xlabel("t [s]")
ax.set_ylabel("dx/dt [m/s]")
ax.legend()
plt.show()